# Predicting relapse risk after discharge



## 1. Data overview

Each row represents one discharge episode, so a resident may appear more than once. The training file contains six-month follow-up outcomes; the test file does not. The target is `relapse_reported = 1` and `engaged_in_recovery = 0`. **Lost-to-follow-up is unknown, not 0.** Those episodes remain in the descriptive analysis but are excluded when fitting and scoring supervised models.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss,
                             confusion_matrix, precision_score, recall_score, fbeta_score)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance


#ROOT = Path.cwd()
#if not (ROOT / 'relapse_train.csv').exists():
#    ROOT = Path('/mnt/data')  # convenient fallback in the shared environment
train = pd.read_csv("relapse_train.csv")
test = pd.read_csv("relapse_test.csv")
print('Train:', train.shape, '| Test:', test.shape)
print('Residents in train:', train.person_id.nunique())
print(train.followup_6m.value_counts())

Train: (1944, 27) | Test: (577, 24)
Residents in train: 1664
followup_6m
engaged_in_recovery    1010
lost_to_follow_up       528
relapse_reported        406
Name: count, dtype: int64


In [ ]:
missing = train.isna().sum()
print('\nColumns with missing values:\n', missing[missing > 0])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
train.followup_6m.value_counts().reindex(
    ['engaged_in_recovery', 'relapse_reported', 'lost_to_follow_up']
).plot(kind='bar', ax=ax[0], color=['#629c88', '#d58a73', '#a0a8b7'])
ax[0].set_xticklabels(['Recovery', 'Relapse', 'Lost'], rotation=0)
ax[0].set_title('Six-month follow-up')
ax[0].set_ylabel('Episodes')
train['los_days'].plot.hist(bins=25, ax=ax[1], color='#629c88')
ax[1].set_title('Length of stay')
ax[1].set_xlabel('Days')
plt.tight_layout()
plt.show()

n = len(train)
known_n = train.followup_6m.ne('lost_to_follow_up').sum()
relapses = train.followup_6m.eq('relapse_reported').sum()
print(f'Follow-up unknown: {n-known_n}/{n} ({(n-known_n)/n:.1%})')
print(f'Observed relapse rate: {relapses}/{known_n} ({relapses/known_n:.1%})')

## 2. Cleaning and feature selection

The main data issues are missing values and inconsistent spellings in `primary_substance`. A few recorded stay lengths disagree with the dates; here I use the date difference. Negative waiting times are treated as missing. Numeric imputation and categorical encoding are learned from the training split, rather than from the complete dataset.

Only information available by discharge is considered. I exclude IDs, raw dates, free-text staff notes, and future information (`aftercare_sessions_6m`, `readmitted_within_12m`). `followup_6m` is used only to create the target. Age and gender are held out for subgroup evaluation; excluding them does not eliminate bias from other features.

In [ ]:
numeric = ['prior_treatment_episodes', 'wait_days', 'los_days', 'phase_reached',
           'incidents_count', 'positive_screens', 'meetings_per_week',
           'co_occurring_mh', 'fee_waived', 'aftercare_plan']
categorical = ['house_id', 'region', 'primary_substance', 'housing_at_intake',
               'referral_source', 'discharge_reason', 'employment_at_discharge']
features = numeric + categorical

def clean(df):
    df = df.copy()
    for c in ['admission_date', 'discharge_date']:
        df[c] = pd.to_datetime(df[c])
    df['los_days'] = (df.discharge_date - df.admission_date).dt.days
    df.loc[df.wait_days < 0, 'wait_days'] = np.nan
    df['primary_substance'] = (df.primary_substance.str.strip().str.lower()
                               .replace({'stimulant': 'stimulants'}))
    for c in categorical:
        df[c] = df[c].fillna('Missing').astype(str)
    for c in ['co_occurring_mh', 'fee_waived', 'aftercare_plan']:
        df[c] = df[c].astype(int)
    return df

train, test = clean(train), clean(test)
train['target'] = train.followup_6m.map({'relapse_reported': 1,
                                          'engaged_in_recovery': 0})
assert train.episode_id.is_unique and test.episode_id.is_unique
assert set(features).issubset(test.columns)
print('Episodes with known outcomes:', train.target.notna().sum())

## 3. Validation strategy

I use three periods: **training before July 2023**, development in January–June 2024, and final evaluation in January–March 2025. The gaps mean that six-month follow-up information from the earlier period could have been available before evaluating the next period. I also remove residents from an earlier split if they appear in a later split, so the same person does not occur in both fitting and validation data.

This is more conservative than a random split and leaves some rows unused. It mainly tests performance on *new* residents; performance on returning residents could differ. The final evaluation set is not used to select the model or threshold.

In [ ]:
fit = train[train.discharge_date < '2023-07-01'].copy()
dev = train[train.discharge_date.between('2024-01-01', '2024-06-30')].copy()
audit = train[train.discharge_date.between('2025-01-01', '2025-03-31')].copy()
fit = fit[~fit.person_id.isin(set(dev.person_id) | set(audit.person_id))]
dev = dev[~dev.person_id.isin(audit.person_id)]
fit, dev, audit = [d[d.target.notna()].copy() for d in (fit, dev, audit)]
assert not set(fit.person_id) & set(dev.person_id)
assert not set(fit.person_id) & set(audit.person_id)
assert not set(dev.person_id) & set(audit.person_id)
print(pd.DataFrame({'split': ['Train', 'Development', 'Final evaluation'],
                    'episodes': [len(fit), len(dev), len(audit)],
                    'relapses': [int(d.target.sum()) for d in (fit, dev, audit)]}))

## 4. Two models

**Logistic regression** is the baseline: it is relatively easy to interpret and works well with small tabular datasets. Histogram gradient boosting is the only additional model: it can capture nonlinear patterns. Both models use the same features and are fitted only on the training split. There is no extensive parameter search, given the small dataset and time limit.

I select a model using development-set average precision (PR-AUC), which is useful when relapses are less common than recoveries. The simpler logistic model is preferred unless boosting improves PR-AUC by at least 0.01.

In [ ]:
def preprocess(scale=False):
    steps = [('impute', SimpleImputer(strategy='median'))]
    if scale:
        steps.append(('scale', StandardScaler()))
    return ColumnTransformer([
        ('num', Pipeline(steps), numeric),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical)
    ])

models = {
    'Logistic regression': Pipeline([
        ('prep', preprocess(scale=True)),
        ('model', LogisticRegression(max_iter=2000, random_state=42))]),
    'Gradient boosting': Pipeline([
        ('prep', preprocess()),
        ('model', HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05,
                   max_leaf_nodes=7, min_samples_leaf=25, l2_regularization=5,
                   early_stopping=False, random_state=42))])
}
rows = []
for name, model in models.items():
    model.fit(fit[features], fit.target.astype(int))
    prob = model.predict_proba(dev[features])[:, 1]
    rows.append({'Model': name,
                 'PR-AUC': average_precision_score(dev.target, prob),
                 'ROC-AUC': roc_auc_score(dev.target, prob),
                 'Brier': brier_score_loss(dev.target, prob)})
comparison = pd.DataFrame(rows).set_index('Model')
print(comparison.round(3))
selected = ('Gradient boosting' if comparison.loc['Gradient boosting', 'PR-AUC'] >=
            comparison.loc['Logistic regression', 'PR-AUC'] + 0.01
            else 'Logistic regression')
model = models[selected]
print('Selected:', selected)

## 5. Choosing a risk threshold

Missing a resident who later relapses may be costly, so I use **F2** (which emphasizes recall) rather than accuracy. However, maximizing F2 alone can flag almost everyone. As an illustrative staffing constraint, I limit alerts to **40% of the development episodes** and choose the threshold with the highest F2 among those that meet the limit. This is not a real staffing commitment; the aftercare team would need to approve a practical capacity before deployment. The threshold is chosen using development data, never the final evaluation outcomes.

In [ ]:
dev_prob = model.predict_proba(dev[features])[:, 1]
thresholds = np.arange(0.05, 0.81, 0.01)
capacity = 0.40  # illustrative maximum fraction of residents flagged on development data
eligible = [t for t in thresholds if (dev_prob >= t).mean() <= capacity]
threshold = float(max(eligible, key=lambda t: fbeta_score(
    dev.target, dev_prob >= t, beta=2, zero_division=0)))
print(f'Development F2 threshold: {threshold:.2f}')

# Evaluate once on the later, untouched period.
y = audit.target.astype(int)
risk = model.predict_proba(audit[features])[:, 1]
flag = (risk >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y, flag, labels=[0, 1]).ravel()
metrics = pd.Series({'ROC-AUC': roc_auc_score(y, risk),
                     'PR-AUC': average_precision_score(y, risk),
                     'Brier score': brier_score_loss(y, risk),
                     'Recall': recall_score(y, flag),
                     'Precision': precision_score(y, flag, zero_division=0),
                     'True positives': tp, 'False positives': fp,
                     'False negatives': fn, 'True negatives': tn})
print(metrics.round(3))

## 6. Calibration

Good ranking does not necessarily mean good probabilities. The Brier score measures squared probability error (lower is better). A calibration plot checks whether residents assigned, for example, around 30% risk actually relapse about 30% of the time. The small evaluation sample means individual bins are noisy; I do not fit a separate calibration model to this holdout.

In [ ]:
observed, predicted = calibration_curve(y, risk, n_bins=5, strategy='quantile')
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
ax[0].plot(predicted, observed, 'o-', label=selected)
ax[0].set(xlabel='Mean predicted risk', ylabel='Observed relapse rate',
          title='Calibration on final evaluation set', xlim=(0, 1), ylim=(0, 1))
ax[0].legend()
ax[1].bar(['TP', 'FP', 'FN', 'TN'], [tp, fp, fn, tn], color='#629c88')
ax[1].set(title=f'Confusion matrix counts (threshold = {threshold:.2f})',
          ylabel='Episodes')
plt.tight_layout()
plt.show()

## 7. Subgroup checks and interpretation

I compare recall and false-positive rates across gender, age, and region, using one threshold for everyone. Small groups can have unstable estimates. These results only include residents whose six-month outcomes were recorded, so differences in follow-up rates may also affect the comparisons.

Permutation importance estimates how much model ranking performance falls when a feature is shuffled. It is useful for discussing model drivers but does not show causation.

In [ ]:
audit_result = audit[['gender', 'age_band', 'region']].copy()
audit_result['target'] = y
audit_result['predicted'] = flag

def subgroup(col):
    out = []
    for group, d in audit_result.groupby(col, dropna=False):
        tn_g, fp_g, fn_g, tp_g = confusion_matrix(
            d.target, d.predicted, labels=[0, 1]).ravel()
        out.append({'variable': col, 'group': str(group), 'n': len(d),
                    'relapses': int(d.target.sum()),
                    'recall': tp_g / (tp_g + fn_g) if tp_g + fn_g else np.nan,
                    'false_positive_rate': fp_g / (fp_g + tn_g) if fp_g + tn_g else np.nan})
    return pd.DataFrame(out)

fairness = pd.concat([subgroup(c) for c in ['gender', 'age_band', 'region']])
print(fairness.round(3).to_string(index=False))
print('\nFollow-up missing by region:')
print(train.assign(lost=train.followup_6m.eq('lost_to_follow_up'))
      .groupby('region').lost.agg(['count', 'mean']).round(3))

importance = permutation_importance(model, audit[features], y,
                                    scoring='average_precision', n_repeats=5,
                                    random_state=42)
imp = pd.Series(importance.importances_mean, index=features).sort_values(ascending=False)
print('\nTop predictors (permutation importance):')
print(imp.head(8).round(3))

## 8. Predictions for the test set

The submission requires one probability and one binary label for each episode. The model and threshold are kept unchanged after final evaluation, so test scores come from the version actually evaluated. No test outcomes were used. As a limitation, this means the final model does not take advantage of every labeled training record.

In [ ]:
test_risk = model.predict_proba(test[features])[:, 1]
predictions = pd.DataFrame({
    'episode_id': test.episode_id,
    'risk_score': test_risk,
    'predicted_relapse': (test_risk >= threshold).astype(int)
})
assert len(predictions) == len(test)
assert predictions.episode_id.is_unique and predictions.risk_score.between(0, 1).all()
assert predictions.notna().all().all()
predictions.to_csv('predictions.csv', index=False)
print(f'Saved {len(predictions)} predictions; {predictions.predicted_relapse.sum()} flagged.')
print(predictions.head().to_string(index=False))

## 9. Discussion and conclusion

On the untouched evaluation period, **logistic regression** achieved **ROC-AUC 0.758**, **PR-AUC 0.471**, and **Brier score 0.141**. At the chosen **0.23** threshold, it identified **17 of 23** recorded relapses, missed **6**, and raised **28** false alarms. This is useful discrimination for an initial experiment, but not enough to establish clinical reliability. Although the development-set alert budget was limited to 40%, the model flagged **302 of 577 (52.3%)** test episodes, showing that workload could increase in later cohorts.

The largest limitation is **lost follow-up** (528/1,944 episodes). Excluding these episodes avoids treating them as recoveries, but the reported performance applies only to residents with observed outcomes. The full-cohort relapse rate could range from **20.9% to 48.0%** under extreme assumptions about the missing outcomes. The subgroup results are also uncertain because some groups have very few events, and missing follow-up differs by region.

In practice, this score should only help staff **offer extra support**, not deny care or replace professional judgment. A prospective pilot should check the workload, missed relapses, false alarms, probability calibration, follow-up completion, and performance differences across demographic groups. Staff must be able to review and override any flag. The dataset is synthetic, and this analysis is a course-style demonstration rather than evidence of clinical readiness.